In [4]:
import pandas as pd
import os

# 读取CSV文件
file_path = r"D:\weibo-search\结果文件\南京地铁\情感因素分析\安全聚类结果.csv"
df = pd.read_csv(file_path, encoding='utf_8_sig')  # 使用utf_8_sig编码防止中文乱码

# 按簇ID分组并转换为字典 {簇ID: [文本列表]}
cluster_dict = df.groupby('簇ID')['文本'].apply(list).to_dict()

# 准备输出内容
output_lines = []
for cluster_id, texts in cluster_dict.items():
    cluster_header = f"\n◆ 簇 {cluster_id} (共 {len(texts)} 条):"
    output_lines.append(cluster_header)
    output_lines.append("-" * len(cluster_header))
    
    # 打印到控制台
    print(cluster_header)
    for i, text in enumerate(texts, 1):
        output_line = f"{i}. {text}"
        output_lines.append(output_line)
        print(output_line)

# 确定输出文件路径
output_dir = os.path.dirname(file_path)
output_file = os.path.join(output_dir, "聚类结果输出.txt")

# 写入文本文件
with open(output_file, 'w', encoding='utf-8') as f:
    f.write("\n".join(output_lines))

print(f"\n聚类结果已保存至: {output_file}")


◆ 簇 1 (共 152 条):
1. 空间宽敞，设施现代，体验舒适
2. 环境舒适，换乘便捷
3. 运行稳，乘车舒适
4. 环境安静，声音悦耳
5. 发车间隔短，乘车舒适，座位充足
6. 声音愉悦，体验舒适
7. 气候宜人，温度舒适
8. 舒适便捷，线路喜爱
9. 舒适度高，温度适宜
10. 车厢空旷，乘坐舒适
11. 舒适体验，优越对比
12. 便捷，舒适
13. 温暖舒适，灯光适宜
14. 温暖舒适，环境宜人
15. 人性化，暖气舒适
16. 凉爽舒适，缓解疲劳
17. 悦耳铃声，熟悉亲切
18. 平稳，舒适
19. 温暖灯光，怀旧氛围
20. 光线舒适，氛围温馨
21. 座位充足，乘坐舒适
22. 座位充足，运行快速
23. 灯光效果，环境明亮
24. 座位充足，环境整洁
25. 平稳，舒适
26. 干净，舒适
27. 熟悉感，归属感，便捷性
28. 环境舒适，氛围清新
29. 安静舒适，空间宽松，氛围和谐
30. 舒适度高，空调适宜，环境清新
31. 舒适安静，运行平稳，设计独特
32. 乘车高效，座位充足，服务暖心
33. 灯光舒适，氛围温馨
34. 舒适便捷，乘坐体验
35. 舒适度高，环境安静，车厢平稳
36. 凉快，舒适
37. 温暖舒适，设施完善
38. 怀旧感，个人回忆，舒适环境
39. 舒适度高，座位充足，设计合理
40. 座位充足，乘车舒适
41. 麦浪香味，气温舒适
42. 开门时间早，设施舒适，改票顺利
43. 舒适便捷，座位设计，城市特色
44. 车厢设计，联名创意，氛围舒适
45. 温暖舒适，温度适宜，环境惬意
46. 清新空气，气味宜人
47. 座位充足，车厢空旷
48. 宽敞，舒适
49. 归属感，舒适感
50. 行驶平稳，乘坐舒适
51. 凉爽舒适，温度适宜
52. 舒适度高，光线柔和
53. 凉爽舒适，温度适宜
54. 怀旧感，轻松氛围
55. 座位充足，乘车舒适，人流量少
56. 温暖，光线舒适
57. 舒适环境，温度适宜
58. 灯光颜色，视觉感受
59. 温暖舒适，灯光宜人
60. 空调给力，温度舒适
61. 座位充足，环境舒适
62. 氛围温馨，光线舒适
63. 氛围感，灯光舒适，食欲提升
64. 便捷舒适，体验良好
65. 座位充足，乘坐舒适
66. 座位充足，体验舒适
67. 凉快舒适，温度适宜
68. 氛围感

In [ ]:
import re
import requests
import json
from collections import defaultdict

# DeepSeek API配置
DEEPSEEK_API_URL = "https://api.deepseek.com/v1/chat/completions"
API_KEY = "DEEPSEEK_API_KEY"  

# 文件路径配置
input_txt = r"D:\weibo-search\结果文件\南京地铁\情感因素分析\聚类结果输出.txt"
output_txt = r"D:\weibo-search\结果文件\南京地铁\情感因素分析\聚类结果_对齐输出.txt"

def parse_txt_file(file_path):
    """解析特殊格式的TXT文件"""
    cluster_dict = defaultdict(list)
    current_cluster = None
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            # 匹配簇标题行，如："◆ 簇 1 (共 6 条):"
            cluster_match = re.match(r'◆ 簇 (\d+) \(共 (\d+) 条\):', line)
            if cluster_match:
                current_cluster = int(cluster_match.group(1))
                continue
                
            # 匹配文本行，如："1. 播报温暖，守护陪伴，情感共鸣"
            text_match = re.match(r'\d+\. (.+)', line)
            if text_match and current_cluster is not None:
                cluster_dict[current_cluster].append(text_match.group(1))
    
    return cluster_dict

def align_semantics(texts):
    """使用DeepSeek API进行语义对齐"""
    prompt = f"""
    请将以下关于地铁服务的乘客反馈提炼为最核心的标准化表述，要求：
    1. 输出仅一条标准化表述
    2. 只保留最本质的需求/问题/感受
    3. 使用"乘客"作为主语
    4. 采用"乘客+核心诉求"的极简结构
    5. 不要包含具体线路、时段等细节

    【输出规范】
    1. 格式：严格遵循"乘客+[诉求类型]+[核心问题]"结构
    2. 长度：不超过20个汉字（含标点）
    3. 诉求类型标注：
       - (需求) 乘客主动提出的改进需求
       - (问题) 乘客反映的现存问题  
       - (评价) 乘客表达的服务评价
    4. 必须包含最高频核心词
    5. 优先选择出现频率最高的诉求类型（服务/拥挤/设施等）

    【优化原则】
    1. 聚焦可行动项（运营方可直接解决的问题）
    2. 保留程度副词（如"过度拥挤"vs"轻微拥挤"）
    3. 区分问题类型（设施/服务/管理等）
    4. 消除模糊表述（如"改善服务"→"增加安检速度"）

    【典型范例】
    好的输出：
    "乘客(需求)增加高峰班次"
    "乘客(问题)闸机故障率高"  
    "乘客(评价)赞赏问询服务"
    "乘客(评价)赞赏暖心毕业祝福"

    差的输出：
    "希望改进"（缺少主语和类型）
    "3号线太挤"（含具体线路）

    待分析文本：
    {texts}

    【最终输出】（只需返回标准表述，不要解释）：
    """  
    
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    
    data = {
        "model": "deepseek-chat",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 100
    }
    
    try:
        response = requests.post(DEEPSEEK_API_URL, headers=headers, json=data, timeout=30)
        response.raise_for_status()
        result = response.json()
        return result['choices'][0]['message']['content'].strip('"\' \n')
    except Exception as e:
        print(f"API调用失败: {e}")
        return "【自动生成】" + texts[0][:50] + "..."  # 失败时返回第一条文本的摘要

# 解析输入文件
cluster_dict = parse_txt_file(input_txt)

# 处理并写入TXT文件
with open(output_txt, 'w', encoding='utf-8') as f:
    # 写入文件头
    f.write("="*60 + "\n")
    f.write("南京地铁情感因素聚类结果 - 语义对齐标准表述\n")
    f.write("="*60 + "\n\n")
    
    for cluster_id, texts in sorted(cluster_dict.items()):
        aligned_text = align_semantics(texts)
        
        # 写入簇信息
        f.write(f"■ 簇ID: {cluster_id} (共{len(texts)}条)\n")
        f.write(f"标准表述: {aligned_text}\n")
        f.write("-"*50 + "\n")
        
        # 写入代表性原始文本（最多5条）
        f.write("代表性原始文本:\n")
        for i, text in enumerate(texts[:5], 1):
            f.write(f"{i}. {text}\n")
        
        f.write("\n" + "="*60 + "\n\n")

print(f"语义对齐结果已成功保存至:\n{output_txt}")

语义对齐结果已成功保存至:
D:\weibo-search\结果文件\南京地铁\情感因素分析\聚类结果_对齐输出.txt
